In [1]:
import pandas as pd
import numpy as np


In [2]:
#game state variable

hitter_data = pd.read_csv("/Users/Coleton/Downloads/modeldata_hitter.csv")

hitter_data["game_state"]=(
    hitter_data["inning"] +
    hitter_data["strikes"]/10 +
    hitter_data["outs_when_up"]/100
).round(5)

hitter_data["game_state"][100]

/var/folders/m_/clvqh96x4nj3lzfkfbbf_4gc0000gn/T/ipykernel_58489/27088947.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hitter_data["game_state"]=(


np.float64(7.11)

In [3]:
hitter_data[["inning","strikes","outs_when_up","game_state"]].iloc[3]

inning          2.00
strikes         1.00
outs_when_up    2.00
game_state      2.12
Name: 3, dtype: float64

In [4]:
game_data = hitter_data.groupby(hitter_data["game_pk"]).size().reset_index(name = "challengeable_pitches")
game_data["challengeable_pitches"].describe()
hitter_data





,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,delta_run_exp_2.y,max_runexp.y,min_runexp.y,N.y,rowid.y,delta,challenge_hitter_lost,challenges_remaining,umpire_hp,game_state
0,FF,2026-09-02,92.7,-2.18,5.55,"Lord, Brad",645277,695418,NaN,called_strike,...,0.172977,0.173,0.172,397,19,0.260977,False,2,Dexter Kelley,2.00
1,ST,2026-09-02,82.4,-2.42,5.43,"Lord, Brad",671739,695418,NaN,called_strike,...,0.077000,0.077,0.077,2757,31,0.127000,False,2,Dexter Kelley,2.01
2,SI,2026-09-02,94.5,-2.18,5.37,"Lord, Brad",669221,695418,NaN,called_strike,...,0.038000,0.038,0.038,6547,49,0.081000,False,2,Dexter Kelley,2.02
3,SL,2026-09-02,86.3,-2.20,5.36,"Lord, Brad",669221,695418,NaN,called_strike,...,0.029000,0.029,0.029,3685,51,0.090000,False,2,Dexter Kelley,2.12
4,FF,2026-09-02,96.0,-2.31,5.42,"Lord, Brad",669221,695418,strikeout,called_strike,...,0.025000,0.025,0.025,2172,53,0.178000,False,2,Dexter Kelley,2.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101760,SL,2026-03-30,84.2,-1.58,5.93,"Morillo, Juan",678009,666661,NaN,called_strike,...,0.024000,0.024,0.024,591,553,0.061000,False,2,Todd Tichenor,7.02
101761,SL,2026-03-30,86.3,-0.50,6.27,"Clarke, Taylor",650402,664199,NaN,called_strike,...,0.064000,0.064,0.064,2205,55,0.116000,False,2,Todd Tichenor,8.02
101762,ST,2026-03-30,81.9,-3.47,4.71,"Sewald, Paul",690993,623149,NaN,called_strike,...,0.085000,0.085,0.085,3836,7,0.141021,False,2,Todd Tichenor,9.00
101763,FF,2026-03-30,92.2,-3.49,4.46,"Sewald, Paul",682985,623149,NaN,called_strike,...,0.077000,0.077,0.077,2757,31,0.127000,False,2,Todd Tichenor,9.01


In [33]:
rng = np.random.default_rng(10404)
n_challengeable = rng.choice(game_data["challengeable_pitches"],size = 10000, replace=True)
delta_dist = rng.choice(hitter_data["delta_run_exp"].dropna(),size = 10000, replace = True)
def game_state_deltas(game_state,df = hitter_data,rng = rng): 
    values = df.loc[
        df["game_state"]==game_state,
        "delta_run_exp"
    ].dropna()
    return rng.choice(values)






In [27]:
##Actual Simulation
rng = np.random.default_rng(10404)
n_challengeable_per_game = hitter_data.groupby("game_pk").size()
valid_states = (hitter_data["delta_run_exp"].notna() & hitter_data["game_state"] < 10)
state_deltas = (
    hitter_data.loc[valid_states].groupby("game_state")["delta_run_exp"].apply(lambda x: x.to_numpy()).to_dict()
)

n_games = 10000
games = []
sampleable_states = np.array(list(state_deltas.keys()))
for game_id in range(n_games):
    n_challengeable = rng.choice(n_challengeable_per_game.values)
    game_states = rng.choice(sampleable_states,
                             size = n_challengeable,
                             replace= True)
    for state in game_states:
        delta = rng.choice(state_deltas[state])
        games.append({
            "game_id": game_id,
            "game_state": state,
            "re_delta": delta
        })
simulation_df = pd.DataFrame(games)
simulation_df = simulation_df.sort_values(
    ["game_state", "game_id"],
    ascending=[False, True]
).reset_index(drop=True)


    

In [20]:
simulation_df

['game_id ', 'game_state', 're_delta']


np.float64(13.12)